## 1.0 Code Setup

**ORGANIZATION AND STRUCTURE:**

- `location` and `dim_date` Dimensions - MySQL

- `product vendor` and `vendor` Dimensions - MongoDB

- `product` and `productinventory` Dimensions - CSV

BATCH contains: csv files (imported directly into python) and json files (uploaded to MongoDB then connected) --> have to be created in MYSQL from adventureworks
STREAMING contains: json files of `productinventory` broken up into 3 instances based on ID (since this is the base dimension table)

Order to import:
1. CSV files from batch (local)
2. JSON files from mongoDB
3. Data direct from MYSQL (location and date) directly to dataframe

Then verify all tables exist (should have 6)

We only need 1 FACT table, so we only need streaming data for 1 set of tables (*bronze*, *silver*, and *gold*)

- *Bronze* Table = Get `productinventory` 

- *Silver* Table = join with `production`, `location`, `vendor`, `vendor product`, `date`

- *Gold* Table = Creates total inventory by location and product

**CREATING "MOCK" DATA:**

Adventureworks was loaded into MySQL. The necessary tables (outlined above) were exported to .json files for MongoDB and .csv files for local storage. Additionally, the date dimension was loaded, with date range changed from the provided code in lab_02c to match adventureworks range vs northwind in class. Lastly, the productinventory data was split into 3 in MySQL. The code files for these are located in the submission.

1.1 Import Packages

In [1]:
import findspark
findspark.init("C:\\spark-3.5.7-bin-hadoop3")
print(findspark.find())

import os
import sys
import json
import time
import pymongo
import certifi
import shutil
import pandas as pd
import numpy as np
import datetime

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

import sqlalchemy
from sqlalchemy import create_engine, text

C:\spark-3.5.7-bin-hadoop3


In [2]:
# Set the working directory to the project folder
os.chdir('C:\\Users\\taran\\OneDrive - University of Virginia\\documents\\DS 2002\\Projects\\Project 2')

1.2 Global Variables

In [ ]:
# --------------------------------------------------------------------------------
# Specify MySQL Server Connection Information
# --------------------------------------------------------------------------------
mysql_args = {
    "host_name" : "localhost",
    "port" : "3306",
    "db_name" : "adventureworks",
    "conn_props" : {
        "user" : "root",
        "password" : "CHANGED FOR UPLOAD",
        "driver" : "com.mysql.cj.jdbc.Driver"
    }
}

# --------------------------------------------------------------------------------
# Specify MongoDB Cluster Connection Information
# --------------------------------------------------------------------------------
mongodb_args = {
    "cluster_location" : "atlas", # "atlas"
    "user_name" : "CHANGED FOR UPLOAD",
    "password" : "CHANGED FOR UPLOAD",
    "cluster_name" : "cluster0",
    "cluster_subnet" : "2ypncz8",
    "db_name" : "adventureworks",
    "collection" : "",
    "null_column_threshold" : 0.5
}

# --------------------------------------------------------------------------------
# Specify Directory Structure for Source Data
# --------------------------------------------------------------------------------
base_dir = os.path.join(os.getcwd(), 'project_data')
data_dir = os.path.join(base_dir, 'adventureworks-data')
batch_dir = os.path.join(data_dir, 'batch')
stream_dir = os.path.join(data_dir, 'streaming')

product_inventory_stream_dir = os.path.join(stream_dir, 'product_inventory')

# --------------------------------------------------------------------------------
# Create Directory Structure for Data Lakehouse Files
# --------------------------------------------------------------------------------
dest_database = "adventureworks_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

product_inventory_output_bronze = os.path.join(database_dir, 'fact_product_inventory', 'bronze')
product_inventory_output_silver = os.path.join(database_dir, 'fact_product_inventory', 'silver')
product_inventory_output_gold = os.path.join(database_dir, 'fact_product_inventory', 'gold')


1.3. Global Functions (Connection)

In [4]:
def get_file_info(path: str):
    file_sizes = []
    modification_times = []

    '''Fetch each item in the directory, and filter out any directories.'''
    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])

    '''Populate lists with the Size and Last Modification DateTime for each file in the directory.'''
    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))

    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name','size','modification_time']
    
    return pd.DataFrame(data=data, columns=column_names)


def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
        
    print(f"The stream has processed {len(query.recentProgress)} batchs")


def remove_directory_tree(path: str):
    '''If it exists, remove the entire contents of a directory structure at a given 'path' parameter's location.'''
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
            
    except Exception as e:
        return f"An error occurred: {e}"
        

def drop_null_columns(df, threshold):
    '''Drop Columns having a percentage of NULL values that exceeds the given 'threshold' parameter value.'''
    columns_with_nulls = [col for col in df.columns if df.filter(df[col].isNull()).count() / df.count() > threshold] 
    df_dropped = df.drop(*columns_with_nulls) 
    
    return df_dropped
    
    
def get_mysql_dataframe(spark_session, sql_query : str, **args):
    '''Create a JDBC URL to the MySQL Database'''
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"
    
    '''Invoke the spark.read.format("jdbc") function to query the database, and fill a DataFrame.'''
    dframe = spark_session.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("driver", args['conn_props']['driver']) \
    .option("user", args['conn_props']['user']) \
    .option("password", args['conn_props']['password']) \
    .option("query", sql_query) \
    .load()
    
    return dframe
    

def get_mongo_uri(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the 'cluster_location' parameter.")
        
    if args['cluster_location'] == "atlas":
        uri = f"mongodb+srv://{args['user_name']}:{args['password']}@"
        uri += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
    else:
        uri = "mongodb://localhost:27017/"

    return uri


def get_spark_conf_args(spark_jars : list, **args):
    jars = ""
    for jar in spark_jars:
        jars += f"{jar}, "
    
    sparkConf_args = {
        "app_name" : "PySpark Adventureworks Data Lakehouse (Medallion Architecture)",
        "worker_threads" : f"local[{int(os.cpu_count()/2)}]",
        "shuffle_partitions" : int(os.cpu_count()),
        "mongo_uri" : get_mongo_uri(**args),
        "spark_jars" : jars[0:-2],
        "database_dir" : sql_warehouse_dir
    }
    
    return sparkConf_args
    

def get_spark_conf(**args):
    sparkConf = SparkConf().setAppName(args['app_name'])\
    .setMaster(args['worker_threads']) \
    .set('spark.driver.memory', '4g') \
    .set('spark.executor.memory', '2g') \
    .set('spark.jars', args['spark_jars']) \
    .set('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector_2.12:3.0.1') \
    .set('spark.mongodb.input.uri', args['mongo_uri']) \
    .set('spark.mongodb.output.uri', args['mongo_uri']) \
    .set('spark.sql.adaptive.enabled', 'false') \
    .set('spark.sql.debug.maxToStringFields', 35) \
    .set('spark.sql.shuffle.partitions', args['shuffle_partitions']) \
    .set('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true') \
    .set('spark.sql.streaming.schemaInference', 'true') \
    .set('spark.sql.warehouse.dir', args['database_dir']) \
    .set('spark.streaming.stopGracefullyOnShutdown', 'true')
    
    return sparkConf


def get_mongo_client(**args):
    '''Get MongoDB Client Connection'''
    mongo_uri = get_mongo_uri(**args)
    if args['cluster_location'] == "atlas":
        client = pymongo.MongoClient(mongo_uri, tlsCAFile=certifi.where())

    elif args['cluster_location'] == "local":
        client = pymongo.MongoClient(mongo_uri)
        
    else:
        raise Exception("A MongoDB Client could not be created.")

    return client
    
    
def set_mongo_collections(mongo_client, db_name : str, data_directory : str, json_files : list):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()
    

def get_mongodb_dataframe(spark_session, **args):
    '''Query MongoDB, and create a DataFrame'''
    dframe = spark_session.read.format("com.mongodb.spark.sql.DefaultSource") \
        .option("database", args['db_name']) \
        .option("collection", args['collection']).load()

    '''Drop the '_id' index column to clean up the response.'''
    dframe = dframe.drop('_id')
    
    '''Call the drop_null_columns() function passing in the dataframe.'''
    dframe = drop_null_columns(dframe, args['null_column_threshold'])
    
    return dframe

1.4 Remove Any Existing Structure

In [5]:
remove_directory_tree(database_dir)

"An error occurred: [WinError 5] Access is denied: 'C:\\\\Users\\\\taran\\\\OneDrive - University of Virginia\\\\documents\\\\DS 2002\\\\Projects\\\\Project 2\\\\spark-warehouse\\\\adventureworks_dlh.db\\\\fact_product_inventory\\\\bronze\\\\_checkpoint\\\\sources\\\\0'"

1.5 Create New Spark Session

In [6]:
worker_threads = f"local[{int(os.cpu_count()/2)}]"

jars = []
mysql_spark_jar = os.path.join(os.getcwd(), "mysql-connector-j-9.1.0", "mysql-connector-j-9.1.0.jar")
mssql_spark_jar = os.path.join(os.getcwd(), "sqljdbc_12.8", "enu", "jars", "mssql-jdbc-12.8.1.jre11.jar")

jars.append(mysql_spark_jar)
#jars.append(mssql_spark_jar)

sparkConf_args = get_spark_conf_args(jars, **mongodb_args)

sparkConf = get_spark_conf(**sparkConf_args)
spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
spark.sparkContext.setLogLevel("OFF")
spark

1.6 Create a New Metadata Database

In [7]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database} CASCADE;")

sql_create_db = f"""
    CREATE DATABASE IF NOT EXISTS {dest_database}
    COMMENT 'DS-2002 Project 2 Database'
    WITH DBPROPERTIES (contains_pii = true, purpose = 'DS-2002 Project 2');
"""
spark.sql(sql_create_db)

DataFrame[]

1.7 Create new MongoDB Database and load JSON files

In [8]:
client = get_mongo_client(**mongodb_args)

json_files = {"vendor_table" : 'vendor_table.json',
              "vendorproduct_table" : 'vendorproduct_table.json',
             }

set_mongo_collections(client, mongodb_args["db_name"], batch_dir, json_files) 

## 2.0 Create Dimensions Tables

### 2.1 Importing Tables from CSV

2.11 Verify Location of source data

In [9]:
get_file_info(batch_dir)

,name,size,modification_time
0,product_table.csv,85216,2025-10-26 17:58:11.343022346
1,productinventory_table.csv,41166,2025-10-26 17:58:22.162152290
2,vendor_table.json,25972,2025-10-26 17:58:35.472940922
3,vendorproduct_table.json,126329,2025-10-26 17:58:47.486377239


2.12 Read in Product Dimension and store in dataframe

In [10]:
spark.sql("DROP TABLE IF EXISTS dim_products")
spark.sql("DROP VIEW IF EXISTS dim_products")

product_csv = os.path.join(batch_dir, 'product_table.csv')
print(product_csv)

df_dim_products = spark.read.format('csv').options(header='true', inferSchema='true').load(product_csv)
df_dim_products.toPandas().head(2)

C:\Users\taran\OneDrive - University of Virginia\documents\DS 2002\Projects\Project 2\project_data\adventureworks-data\batch\product_table.csv


,ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
0,1,Adjustable Race,AR-5381,0,0,NULL,1000,750,0.0,0.0,...,NULL,NULL,NULL,NULL,NULL,1998-06-01,NULL,NULL,...,2004-03-11 10:01:36
1,2,Bearing Ball,BA-8327,0,0,NULL,1000,750,0.0,0.0,...,NULL,NULL,NULL,NULL,NULL,1998-06-01,NULL,NULL,...,2004-03-11 10:01:36


In [11]:
df_dim_products = df_dim_products.withColumnRenamed("ProductID", "product_id")

df_dim_products.createOrReplaceTempView("dim_products")
sql_dim_products = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY product_id) AS product_key
    FROM dim_products;
"""
df_dim_products = spark.sql(sql_dim_products)

ordered_columns = ['product_key', 'product_id', 'name', 'ProductNumber', 'MakeFlag', 'FinishedGoodsFlag', 'Color', 'SafetyStockLevel', 'ReorderPoint', 'StandardCost', 'ListPrice',
                   'Size','SizeUnitMeasureCode','WeightUnitMeasureCode', 'Weight', 'DaystoManufacture', 'ProductLine', 'Class', 'Style', 'ProductSubcategoryID', 'ProductModelID',
                   'SellStartDate', 'SellEndDate','DiscontinuedDate', 'rowguid', 'ModifiedDate']

df_dim_products = df_dim_products[ordered_columns]
df_dim_products.toPandas().head(2)

,product_key,product_id,name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,...,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
0,1,1,Adjustable Race,AR-5381,0,0,NULL,1000,750,0.0,...,NULL,NULL,NULL,NULL,NULL,1998-06-01,NULL,NULL,...,2004-03-11 10:01:36
1,2,2,Bearing Ball,BA-8327,0,0,NULL,1000,750,0.0,...,NULL,NULL,NULL,NULL,NULL,1998-06-01,NULL,NULL,...,2004-03-11 10:01:36


In [12]:
df_dim_products.write.saveAsTable(f"{dest_database}.dim_products", mode='overwrite')

spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_products;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_products LIMIT 5;").toPandas()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|         product_key|      int|   NULL|
|          product_id|      int|   NULL|
|                name|   string|   NULL|
|       ProductNumber|   string|   NULL|
|            MakeFlag|      int|   NULL|
|   FinishedGoodsFlag|      int|   NULL|
|               Color|   string|   NULL|
|    SafetyStockLevel|      int|   NULL|
|        ReorderPoint|      int|   NULL|
|        StandardCost|   double|   NULL|
|           ListPrice|   double|   NULL|
|                Size|   string|   NULL|
| SizeUnitMeasureCode|   string|   NULL|
|WeightUnitMeasure...|   string|   NULL|
|              Weight|   string|   NULL|
|   DaystoManufacture|      int|   NULL|
|         ProductLine|   string|   NULL|
|               Class|   string|   NULL|
|               Style|   string|   NULL|
|ProductSubcategoryID|   string|   NULL|
+--------------------+---------+-------+
only showing top

,product_key,product_id,name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,...,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
0,1,1,Adjustable Race,AR-5381,0,0,NULL,1000,750,0.0,...,NULL,NULL,NULL,NULL,NULL,1998-06-01,NULL,NULL,...,2004-03-11 10:01:36
1,2,2,Bearing Ball,BA-8327,0,0,NULL,1000,750,0.0,...,NULL,NULL,NULL,NULL,NULL,1998-06-01,NULL,NULL,...,2004-03-11 10:01:36
2,3,3,BB Ball Bearing,BE-2349,1,0,NULL,800,600,0.0,...,NULL,NULL,NULL,NULL,NULL,1998-06-01,NULL,NULL,...,2004-03-11 10:01:36
3,4,4,Headset Ball Bearings,BE-2908,0,0,NULL,800,600,0.0,...,NULL,NULL,NULL,NULL,NULL,1998-06-01,NULL,NULL,...,2004-03-11 10:01:36
4,5,316,Blade,BL-2036,1,0,NULL,800,600,0.0,...,NULL,NULL,NULL,NULL,NULL,1998-06-01,NULL,NULL,...,2004-03-11 10:01:36


2.13 Read in Product Inventory and store in dataframe

In [13]:
spark.sql("DROP TABLE IF EXISTS dim_product_inventory")
spark.sql("DROP VIEW IF EXISTS dim_product_inventory")

product_csv = os.path.join(batch_dir, 'productinventory_table.csv')
print(product_csv)

df_dim_product_inventory = spark.read.format('csv').options(header='true', inferSchema='true').load(product_csv)
df_dim_product_inventory.toPandas().head(2)

C:\Users\taran\OneDrive - University of Virginia\documents\DS 2002\Projects\Project 2\project_data\adventureworks-data\batch\productinventory_table.csv


,ProductID,LocationID,Shelf,Bin,Quantity,rowguid,ModifiedDate
0,1,1,A,1,408,...,2004-09-08
1,1,6,B,5,324,...,2004-09-08


In [14]:
df_dim_product_inventory = df_dim_product_inventory.withColumnRenamed("ProductID", "product_id")

df_dim_product_inventory.createOrReplaceTempView("dim_product_inventory")
sql_dim_product_inventory = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY product_id) AS product_key
    FROM dim_product_inventory;
"""
df_dim_product_inventory = spark.sql(sql_dim_product_inventory)

ordered_columns = ['product_key', 'product_id', 'locationID', 'Shelf', 'Bin', 'Quantity', 'rowguid', 'ModifiedDate']

df_dim_product_inventory = df_dim_product_inventory[ordered_columns]
df_dim_product_inventory.toPandas().head(2)

,product_key,product_id,locationID,Shelf,Bin,Quantity,rowguid,ModifiedDate
0,1,1,1,A,1,408,...,2004-09-08
1,2,1,6,B,5,324,...,2004-09-08


In [15]:
df_dim_product_inventory.write.saveAsTable(f"{dest_database}.dim_product_inventory", mode='overwrite')

spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_product_inventory;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_product_inventory LIMIT 5;").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|         product_key|                 int|   NULL|
|          product_id|                 int|   NULL|
|          locationID|                 int|   NULL|
|               Shelf|              string|   NULL|
|                 Bin|                 int|   NULL|
|            Quantity|                 int|   NULL|
|             rowguid|              string|   NULL|
|        ModifiedDate|           timestamp|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|  adventureworks_dlh|       |
|               Table|dim_product_inven...|       |
|        Created Time|Thu Dec 18 17:49:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 3.5.7|       |
|           

,product_key,product_id,locationID,Shelf,Bin,Quantity,rowguid,ModifiedDate
0,1,1,1,A,1,408,...,2004-09-08
1,2,1,6,B,5,324,...,2004-09-08
2,3,1,50,A,5,353,...,2004-09-08
3,4,2,1,A,2,427,...,2004-09-08
4,5,2,6,B,1,318,...,2004-09-08


### 2.2 Importing Tables from MongoDB

2.21 Read in Vendor and store in dataframe

In [16]:
mongodb_args["collection"] = "vendor_table"

df_dim_vendor = get_mongodb_dataframe(spark, **mongodb_args)
df_dim_vendor.toPandas().head(2)

,AccountNumber,ActiveFlag,CreditRating,ModifiedDate,Name,PreferredVendorStatus,VendorID
0,INTERNAT0001,1,1,2002-02-25 00:00:00,International,1,1
1,ELECTRON0002,1,1,2002-02-17 00:00:00,Electronic Bike Repair & Supplies,1,2


In [17]:
df_dim_vendor = df_dim_vendor.withColumnRenamed("VendorID", "vendor_id")

df_dim_vendor.createOrReplaceTempView("dim_vendor")
sql_dim_vendor = f"""
    SELECT *, ROW_NUMBER() OVER (ORDER BY vendor_id) AS vendor_key
    FROM dim_vendor;
"""
df_dim_vendor = spark.sql(sql_dim_vendor)

ordered_columns = ['vendor_key', 'vendor_id', 'Name', 'AccountNumber', 'ActiveFlag', 'PreferredVendorStatus', 'CreditRating', 'ModifiedDate']

df_dim_vendor = df_dim_vendor[ordered_columns]
df_dim_vendor.toPandas().head(2)

,vendor_key,vendor_id,Name,AccountNumber,ActiveFlag,PreferredVendorStatus,CreditRating,ModifiedDate
0,1,1,International,INTERNAT0001,1,1,1,2002-02-25 00:00:00
1,2,2,Electronic Bike Repair & Supplies,ELECTRON0002,1,1,1,2002-02-17 00:00:00


In [18]:
df_dim_vendor.write.saveAsTable(f"{dest_database}.dim_vendor", mode='overwrite')

spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_vendor;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_vendor LIMIT 5;").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|          vendor_key|                 int|   NULL|
|           vendor_id|                 int|   NULL|
|                Name|              string|   NULL|
|       AccountNumber|              string|   NULL|
|          ActiveFlag|              string|   NULL|
|PreferredVendorSt...|              string|   NULL|
|        CreditRating|                 int|   NULL|
|        ModifiedDate|              string|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|  adventureworks_dlh|       |
|               Table|          dim_vendor|       |
|        Created Time|Thu Dec 18 17:49:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 3.5.7|       |
|           

,vendor_key,vendor_id,Name,AccountNumber,ActiveFlag,PreferredVendorStatus,CreditRating,ModifiedDate
0,1,1,International,INTERNAT0001,1,1,1,2002-02-25 00:00:00
1,2,2,Electronic Bike Repair & Supplies,ELECTRON0002,1,1,1,2002-02-17 00:00:00
2,3,3,"Premier Sport, Inc.",PREMIER0001,1,1,1,2002-03-05 00:00:00
3,4,4,Comfort Road Bicycles,COMFORT0001,1,1,1,2002-01-24 00:00:00
4,5,5,Metro Sport Equipment,METROSP0001,1,1,1,2002-03-01 00:00:00


2.22 Read in Vendor Product and store in dataframe

In [19]:
mongodb_args["collection"] = "vendorproduct_table"

df_dim_vendor_product = get_mongodb_dataframe(spark, **mongodb_args)
df_dim_vendor_product.toPandas().head(2)

,AverageLeadTime,LastReceiptCost,LastReceiptDate,MaxOrderQty,MinOrderQty,ModifiedDate,ProductID,StandardPrice,UnitMeasureCode,VendorID
0,17,50.2635,2001-09-29 00:00:00,5,1,2001-09-29 00:00:00,1,47.87,CS,83
1,19,41.9160,2001-09-29 00:00:00,5,1,2001-09-29 00:00:00,2,39.92,CTN,57


In [20]:
df_dim_vendor_product = df_dim_vendor_product.withColumnRenamed("vendorID", "vendor_id")
df_dim_vendor_product = df_dim_vendor_product.withColumnRenamed("productID", "product_id")

ordered_columns = ['product_id', 'vendor_id', 'StandardPrice', 'MinOrderQty', 'MaxOrderQty','LastReceiptCost',
                   'AverageLeadTime','UnitMeasureCode', 'LastReceiptDate', 'ModifiedDate']

df_dim_vendor_product = df_dim_vendor_product[ordered_columns]
df_dim_vendor_product.toPandas().head(2)

,product_id,vendor_id,StandardPrice,MinOrderQty,MaxOrderQty,LastReceiptCost,AverageLeadTime,UnitMeasureCode,LastReceiptDate,ModifiedDate
0,1,83,47.87,1,5,50.2635,17,CS,2001-09-29 00:00:00,2001-09-29 00:00:00
1,2,57,39.92,1,5,41.9160,19,CTN,2001-09-29 00:00:00,2001-09-29 00:00:00


In [21]:
df_dim_vendor_product.write.saveAsTable(f"{dest_database}.dim_vendor_product", mode='overwrite')

spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_vendor_product;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_vendor_product LIMIT 5;").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|          product_id|                 int|   NULL|
|           vendor_id|                 int|   NULL|
|       StandardPrice|              double|   NULL|
|         MinOrderQty|                 int|   NULL|
|         MaxOrderQty|                 int|   NULL|
|     LastReceiptCost|              double|   NULL|
|     AverageLeadTime|                 int|   NULL|
|     UnitMeasureCode|              string|   NULL|
|     LastReceiptDate|              string|   NULL|
|        ModifiedDate|              string|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|  adventureworks_dlh|       |
|               Table|  dim_vendor_product|       |
|        Created Time|Thu Dec 18 17:49:...|       |
|         La

,product_id,vendor_id,StandardPrice,MinOrderQty,MaxOrderQty,LastReceiptCost,AverageLeadTime,UnitMeasureCode,LastReceiptDate,ModifiedDate
0,1,83,47.87,1,5,50.2635,17,CS,2001-09-29 00:00:00,2001-09-29 00:00:00
1,2,57,39.92,1,5,41.9160,19,CTN,2001-09-29 00:00:00,2001-09-29 00:00:00
2,4,85,54.31,1,5,57.0255,17,CTN,2001-09-29 00:00:00,2001-09-29 00:00:00
3,317,50,28.17,100,1000,29.5785,19,EA,2001-09-29 00:00:00,2001-09-29 00:00:00
4,317,84,25.77,100,1000,27.0585,17,EA,2001-09-25 00:00:00,2001-09-25 00:00:00


### 2.3 Importing Tables from MySQL

2.31 Read in Date Dimension and store in dataframe

In [22]:
sql_dim_date = f"SELECT * FROM {mysql_args['db_name']}.dim_date"
df_dim_date = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)

df_dim_date.write.saveAsTable(f"{dest_database}.dim_date", mode="overwrite")

spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_date;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_date LIMIT 2").toPandas()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|            date_key|      int|   NULL|
|           full_date|     date|   NULL|
|           date_name| char(11)|   NULL|
|        date_name_us| char(11)|   NULL|
|        date_name_eu| char(11)|   NULL|
|         day_of_week|  tinyint|   NULL|
|    day_name_of_week| char(10)|   NULL|
|        day_of_month|  tinyint|   NULL|
|         day_of_year|      int|   NULL|
|     weekday_weekend| char(10)|   NULL|
|        week_of_year|  tinyint|   NULL|
|          month_name| char(10)|   NULL|
|       month_of_year|  tinyint|   NULL|
|is_last_day_of_month|  char(1)|   NULL|
|    calendar_quarter|  tinyint|   NULL|
|       calendar_year|      int|   NULL|
| calendar_year_month| char(10)|   NULL|
|   calendar_year_qtr| char(10)|   NULL|
|fiscal_month_of_year|  tinyint|   NULL|
|      fiscal_quarter|  tinyint|   NULL|
+--------------------+---------+-------+
only showing top

,date_key,full_date,date_name,date_name_us,date_name_eu,day_of_week,day_name_of_week,day_of_month,day_of_year,weekday_weekend,...,is_last_day_of_month,calendar_quarter,calendar_year,calendar_year_month,calendar_year_qtr,fiscal_month_of_year,fiscal_quarter,fiscal_year,fiscal_year_month,fiscal_year_qtr
0,19950101,1995-01-01,1995/01/01,01/01/1995,01/01/1995,1,Sunday,1,1,Weekend,...,N,1,1995,1995-01,1995Q1,7,3,1995,1995-07,1995Q3
1,19950102,1995-01-02,1995/01/02,01/02/1995,02/01/1995,2,Monday,2,2,Weekday,...,N,1,1995,1995-01,1995Q1,7,3,1995,1995-07,1995Q3


2.32 Read in Location Dimension and store in dataframe

In [23]:
sql_dim_location = f"SELECT * FROM {mysql_args['db_name']}.location"
df_dim_location = get_mysql_dataframe(spark, sql_dim_location, **mysql_args)

df_dim_location.createOrReplaceTempView("dim_location_temp")

sql_with_key = """
SELECT
    ROW_NUMBER() OVER (ORDER BY locationID) AS location_key,
    locationID,
    Name,
    CostRate,
    Availability,
    ModifiedDate
FROM dim_location_temp
"""

df_dim_location = spark.sql(sql_with_key)

df_dim_location.head(2)


[Row(location_key=1, locationID=1, Name='Tool Crib', CostRate=0.0, Availability=Decimal('0.00'), ModifiedDate=datetime.datetime(1998, 6, 1, 0, 0)),
 Row(location_key=2, locationID=2, Name='Sheet Metal Racks', CostRate=0.0, Availability=Decimal('0.00'), ModifiedDate=datetime.datetime(1998, 6, 1, 0, 0))]

In [24]:
df_dim_location = df_dim_location.withColumnRenamed("locationID", "location_id")

ordered_columns = ['location_key', 'location_id', 'name', 'CostRate', 'Availability', 'ModifiedDate']

df_dim_location = df_dim_location[ordered_columns]
df_dim_location.toPandas().head(2)

,location_key,location_id,name,CostRate,Availability,ModifiedDate
0,1,1,Tool Crib,0.0,0.00,1998-06-01
1,2,2,Sheet Metal Racks,0.0,0.00,1998-06-01


In [25]:
df_dim_location.write.saveAsTable(f"{dest_database}.dim_location", mode="overwrite")

spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_location;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_location LIMIT 2").toPandas()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|        location_key|                 int|   NULL|
|         location_id|                 int|   NULL|
|                name|         varchar(50)|   NULL|
|            CostRate|              double|   NULL|
|        Availability|        decimal(8,2)|   NULL|
|        ModifiedDate|           timestamp|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|  adventureworks_dlh|       |
|               Table|        dim_location|       |
|        Created Time|Thu Dec 18 17:49:...|       |
|         Last Access|             UNKNOWN|       |
|          Created By|         Spark 3.5.7|       |
|                Type|             MANAGED|       |
|            Provider|             parquet|       |
|           

,location_key,location_id,name,CostRate,Availability,ModifiedDate
0,1,1,Tool Crib,0.0,0.00,1998-06-01
1,2,2,Sheet Metal Racks,0.0,0.00,1998-06-01


### 2.4 Verify All Dimension Tables (6)

In [26]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

,namespace,tableName,isTemporary
0,adventureworks_dlh,dim_date,False
1,adventureworks_dlh,dim_location,False
2,adventureworks_dlh,dim_product_inventory,False
3,adventureworks_dlh,dim_products,False
4,adventureworks_dlh,dim_vendor,False
5,adventureworks_dlh,dim_vendor_product,False
6,,dim_location_temp,True
7,,dim_product_inventory,True
8,,dim_products,True
9,,dim_vendor,True


## 3.0 Create Fact Table

### 3.1 Verify Source Location

In [27]:
get_file_info(product_inventory_stream_dir)

,name,size,modification_time
0,adventureworks_product_inventory_01.json,167805,2025-12-18 17:16:24.348392725
1,adventureworks_product_inventory_02.json,167938,2025-12-18 17:16:47.689213037
2,adventureworks_product_inventory_03.json,169657,2025-12-18 17:17:05.596416473


### 3.2 Create Bronze Table

3.21 Read JSON files into a Stream

In [28]:
df_product_inventory_bronze = (
    spark.readStream \
    .option("schemaLocation", product_inventory_output_bronze) \
    .option("maxFilesPerTrigger", 1) \
    .option("multiLine", "true") \
    .json(product_inventory_stream_dir)
)

df_product_inventory_bronze.isStreaming

True

3.22 Write Streaming Data to Parquet File

In [ ]:
product_inventory_checkpoint_bronze = os.path.join(product_inventory_output_bronze, '_checkpoint')

product_inventory_bronze_query = (
    df_product_inventory_bronze
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file", input_file_name())
    
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("product_inventory_bronze")
    .trigger(availableNow = True) \
    .option("checkpointLocation", product_inventory_checkpoint_bronze) \
    .option("compression", "snappy") \
    .start(product_inventory_output_bronze)
)

3.23 Unit Test

In [30]:
print(f"Query ID: {product_inventory_bronze_query.id}")
print(f"Query Name: {product_inventory_bronze_query.name}")
print(f"Query Status: {product_inventory_bronze_query.status}")

Query ID: 0c6a5b42-c26c-4f00-a998-4a9059b14df3
Query Name: product_inventory_bronze
Query Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


3.24 Await Termination

In [31]:
product_inventory_bronze_query.awaitTermination()

### 3.3 Create Silver Table

3.31 Create Necessary Keys

In [57]:
df_dim_modified_date = df_dim_date.select(col("date_key").alias("modified_date_key"), col("full_date").alias("modified_full_date"))
df_dim_last_receipt_date = df_dim_date.select(col("date_key").alias("last_receipt_date_key"), col("full_date").alias("last_receipt_full_date"))

3.32 Define Silver Query to Join Streaming with Batch Data

In [58]:
fact = (
    spark.readStream
         .format("parquet")
         .load(product_inventory_output_bronze)
         .alias("fact")
)

mod_date = df_dim_modified_date.alias("mod_date")
rec_date = df_dim_last_receipt_date.alias("rec_date")
pi = df_dim_product_inventory.alias("pi")
prod = df_dim_products.alias("prod")
loc = df_dim_location.alias("loc")
vp = df_dim_vendor_product.alias("vp")
ven = df_dim_vendor.alias("ven")

In [59]:
df_product_inventory_silver = (fact \
    .join(mod_date, col("mod_date.modified_full_date").cast(DateType()) == col("fact.ModifiedDate").cast(DateType()), "inner") \
    .join( rec_date, col("rec_date.last_receipt_full_date").cast(DateType()) == col("fact.LastReceiptDate").cast(DateType()), "inner") \
    .join( pi, col("pi.product_id") == col("fact.productID").cast(IntegerType()), "inner") \
    .join( prod, col("prod.product_key") == col("fact.productID").cast(IntegerType()), "left") \
    .join( loc, col("loc.location_key") == col("fact.locationID").cast(IntegerType()), "left") \
    .join( vp, col("vp.product_id") == col("fact.productID").cast(IntegerType()), "left") \
    .join( ven, col("ven.vendor_key") == col("fact.vendorID").cast(IntegerType()), "inner") \
)

df_product_inventory_silver = df_product_inventory_silver.select(
    col("pi.product_id").cast(LongType()).alias("product_id"), \
    col("loc.location_id").cast(LongType()).alias("location_id"), \
    col("ven.vendor_id").cast(LongType()).alias("vendor_id"), \

    col("pi.product_key").cast(LongType()).alias("product_key"), \
    col("loc.location_key").cast(LongType()).alias("location_key"), \
    col("ven.vendor_key").cast(LongType()).alias("vendor_key"), \
    col("mod_date.modified_date_key").cast(LongType()).alias("modified_date_key"), \

    col("fact.Shelf"), \
    col("fact.Bin"), \
    col("fact.Quantity"), \

    col("prod.Size"), \
    col("prod.Weight"), \

    col("loc.CostRate"), \
    col("loc.Availability"), \

    col("vp.AverageLeadTime"), \
    col("ven.PreferredVendorStatus"), \
    col("vp.MinOrderQty"), \
    col("vp.MaxOrderQty"), \
)

In [60]:
df_product_inventory_silver.isStreaming

True

In [61]:
df_product_inventory_silver.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- location_id: long (nullable = true)
 |-- vendor_id: long (nullable = true)
 |-- product_key: long (nullable = false)
 |-- location_key: long (nullable = true)
 |-- vendor_key: long (nullable = false)
 |-- modified_date_key: long (nullable = true)
 |-- Shelf: string (nullable = true)
 |-- Bin: long (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Size: string (nullable = true)
 |-- Weight: string (nullable = true)
 |-- CostRate: double (nullable = true)
 |-- Availability: decimal(8,2) (nullable = true)
 |-- AverageLeadTime: integer (nullable = true)
 |-- PreferredVendorStatus: string (nullable = true)
 |-- MinOrderQty: integer (nullable = true)
 |-- MaxOrderQty: integer (nullable = true)



3.33 Write Transformed Streaming Data to Lakehouse

In [62]:
product_inventory_checkpoint_silver = os.path.join(product_inventory_output_silver, '_checkpoint')

product_inventory_silver_query = (
    df_product_inventory_silver.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("product_inventory_silver")
    .trigger(availableNow = True) \
    .option("checkpointLocation", product_inventory_checkpoint_silver) \
    .option("compression", "snappy") \
    .start(product_inventory_output_silver)
)

3.34 Unit Test

In [63]:
print(f"Query ID: {product_inventory_silver_query.id}")
print(f"Query Name: {product_inventory_silver_query.name}")
print(f"Query Status: {product_inventory_silver_query.status}")

Query ID: b3c59fae-e1fc-4a47-9314-e756c2f97ace
Query Name: product_inventory_silver
Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


3.35 Await Termination

In [64]:
product_inventory_silver_query.awaitTermination()

### 3.4 Create Gold Table

3.41 Create Aggregation Query

Aggregation Query is observing (theoretically) product inventory by product and location since last modified date (or multiple modified dates...aka over time)

In [66]:
df_product_inventory_gold = spark.readStream.format("parquet").load(product_inventory_output_silver) \
    .groupBy("modified_date_key", "product_id", "location_id") \
    .agg(
        sum("Quantity").alias("total_quantity"),
        avg("CostRate").alias("avg_cost_rate"),
        avg("Availability").alias("avg_availability")
    )

In [67]:
df_product_inventory_gold.printSchema()

root
 |-- modified_date_key: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- location_id: long (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- avg_cost_rate: double (nullable = true)
 |-- avg_availability: decimal(12,6) (nullable = true)



3.42 Write Streaming Data to Parquet File as "Complete"

In [68]:
product_inventory_gold_query = (
    df_product_inventory_gold.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("fact_product_inventory")
    .start()
)

In [69]:
wait_until_stream_is_ready(product_inventory_gold_query, 1)

The stream has processed 1 batchs


3.43 Query Gold Table from Memory

In [70]:
df_fact_product_inventory = spark.sql("SELECT * FROM fact_product_inventory")
df_fact_product_inventory.printSchema()

root
 |-- modified_date_key: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- location_id: long (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- avg_cost_rate: double (nullable = true)
 |-- avg_availability: decimal(12,6) (nullable = true)



3.44 Create Final Selection

In [85]:
# Aggregate total quantity by product and location
df_fact_product_inventory_final = (
    df_fact_product_inventory
    .groupBy("location_id", "product_id", "modified_date_key")
    .agg({"total_quantity": "sum"})
    .withColumnRenamed("sum(total_quantity)", "total_quantity")
)

# Add dimension names after aggregation
df_fact_product_inventory_final = (
    df_fact_product_inventory_final
    .join( df_dim_location.select(col("location_id"), col("Name").alias("location_name")), on="location_id", how="left")
    .join( df_dim_products.select(col("product_id"), col("Name").alias("product_name")), on="product_id", how="left")
    .join( df_dim_modified_date.select(col("modified_date_key"), col("modified_full_date")), on="modified_date_key", how="left")
    .select(
        col("location_id").cast("integer"),
        col("product_id").cast("integer"),
        col("total_quantity").cast("integer"),
        col("location_name"),
        col("product_name"),
        col("modified_date_key"),
        col("modified_full_date"),
    )
    .orderBy("location_id", "product_id")
)


3.45 Load and Display Final Results

In [86]:
df_fact_product_inventory_final.write.saveAsTable(f"{dest_database}.fact_product_inventory_final", mode="overwrite")
spark.sql(f"SELECT * FROM {dest_database}.fact_product_inventory_final").toPandas()

,location_id,product_id,total_quantity,location_name,product_name,modified_date_key,modified_full_date
0,3.0,495,588,Paint Shop,Paint - Blue,20040911,2004-09-11
1,3.0,496,360,Paint Shop,Paint - Yellow,20040911,2004-09-11
2,4.0,492,168,Paint Storage,Paint - Black,20040911,2004-09-11
3,4.0,493,288,Paint Storage,Paint - Red,20040911,2004-09-11
4,4.0,494,144,Paint Storage,Paint - Silver,20040911,2004-09-11
...,...,...,...,...,...,...,...
584,NaN,441,6144,None,Lock Nut 17,20040830,2004-08-30
585,NaN,442,6084,None,Lock Nut 7,20040830,2004-08-30
586,NaN,443,6024,None,Lock Nut 8,20040904,2004-09-04
587,NaN,444,5964,None,Lock Nut 9,20040904,2004-09-04


### 3.5 Stop Spark Session

In [87]:
spark.stop()